# Sky Foreground vs. Focal-Plane Radius on the Corner WFS (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-06-12
**Last Modified:** 2026-06-12
**Status:** In Progress
**Keywords:** WFS, corner wavefront sensor, sky background, focal plane radius, AOS, galactic latitude

## Description

Study the **sky-foreground level as a function of focal-plane radius** on the eight
Rubin LSST Camera corner wavefront sensors (CWFS).

For each selected visit the notebook produces **four PDF pages**:
1. The eight corner-WFS images with a sky-tuned stretch (as in
   `wfs_corner_postisr_visualize_v1`).
2. The same images with donut/star pixels **masked** (highlighted) — a diagnostic to
   confirm that only sky remains in the unmasked regions. By default the threshold is
   applied to the **residual after subtracting the radial sky model**
   (`mask_method="radial"`), so the strong radial sky gradient is not itself masked;
   set `mask_method="global"` for a single mean+Nσ threshold over the whole chip.
3. The eight curves of **clipped-mean sky flux vs. focal-plane radius** (coarse bins,
   full radial extent).
4. A **zoomed, normalised** version: finer radius bins out to the donut-acceptance
   radius (1.75° ≈ 310.5 mm), with each CCD's curve normalised to 1.0 over
   295–310 mm — to reveal coherent radial structure across the CCDs.

To minimise donut contamination, only visits at **galactic latitude |b| > `b_min`**
are used (RA/Dec come from the butler exposure record `tracking_ra`/`tracking_dec`,
converted to galactic coordinates with `astropy`). The focal-plane position of every
pixel is computed once per detector with `common.camera_utils.pixel_to_focal` (the
geometry is identical for all images).

**Output:** A multi-page PDF (4 pages/visit) in `wfs/output/`.

**Based on:** `wfs_corner_postisr_visualize_v1.ipynb`; AOS `cwfs` reductions in
`LSSTCam/runs/aos/cwfs/danish_1_0/wep_17_3_0/dv_4_2_0/bin_x2` (repo `/repo/main`).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-12 | Aaron Roodman | Initial version: sky foreground vs focal-plane radius, with |b| cut, masking diagnostic, and radial profiles |
| 2026-06-12 | Aaron Roodman | Default masking now uses the residual after subtracting the radial sky model (mask_method="radial"); "global" kept as an option |
| 2026-06-12 | Aaron Roodman | Added page 4: fine-binned (0.25 mm) profile out to the 1.75° donut radius (310.5 mm), normalised per CCD over 295–310 mm |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Select High-Galactic-Latitude Visits](#select)
6. [Focal-Plane Radius Maps](#radius)
7. [Analysis & PDF (4 pages per visit)](#analysis)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

butler_repo = "/repo/main"
collection = "LSSTCam/runs/aos/cwfs/danish_1_0/wep_17_3_0/dv_4_2_0/bin_x2"
dataset_type = "post_isr_image"
instrument = "LSSTCam"

# Night to inspect (day_obs). Nights with the most |b|>30 visits:
#   20260327 (132), 20260325 (84), 20260409 (58), 20260402 (54), 20260404 (49)
day_obs = 20260327

# Galactic-latitude cut: keep only visits with |b| > b_min (fewer stars/donuts).
b_min = 30.0

# How many of the selected (high-b) visits to render, starting at start_index.
# max_visits = None -> all high-b visits on the night.
max_visits = 3
start_index = 0

# The eight corner wavefront-sensor half-chips (detector id -> raft/sensor name).
WFS_DETECTORS = {
    191: "R00_SW0", 192: "R00_SW1",   # R00 raft corner
    195: "R04_SW0", 196: "R04_SW1",   # R04 raft corner
    199: "R40_SW0", 200: "R40_SW1",   # R40 raft corner
    203: "R44_SW0", 204: "R44_SW1",   # R44 raft corner
}

# ---- Sigma clipping ----------------------------------------------------
# clip_sigma is used for (a) the sigma-clipped mean in each radial bin and
# (b) the mask threshold.
clip_sigma = 3.0
clip_maxiters = 5
radial_bin_mm = 2.0              # width of focal-plane radius bins (mm), page 3

# Masking method for page 2:
#   "radial" : threshold the residual (image - radial sky model) at clip_sigma*std
#              -> isolates donuts/stars without masking the radial sky gradient
#   "global" : single threshold mean + clip_sigma*std over the whole chip
mask_method = "radial"

# ---- Page 4: zoomed/normalised radial profile --------------------------
# Donuts are only accepted out to 1.75 deg; with 0.2 arcsec / 10 micron this is a
# physical focal-plane radius of 310.5 mm.
r_max_donut = 310.5              # mm; outer radius for the page-4 profile
fine_bin_mm = 0.25               # finer radius bins for page 4
norm_r_lo, norm_r_hi = 295.0, 310.0   # per-CCD normalisation window (set mean -> 1.0)

# ---- Display stretch (pages 1 & 2) -------------------------------------
stretch_mode = "sky"             # "sky" | "zscale" | "percentile"
sky_lo_sigma = 2.0
sky_hi_sigma = 8.0
zscale_contrast = 0.25
zscale_nsamples = 100_000
pct_low, pct_high = 1.0, 99.0
asinh_a = 0.1
cmap = "gray"
mask_color = "red"               # colour for masked (donut/star) pixels

# ---- Layout / output ---------------------------------------------------
panel_width = 5.0
title_fontsize = 9
output_dir = "output"
output_pdf = None                # None -> output/wfs_sky_radius_{day_obs}.pdf
dpi = 150


<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.backends.backend_pdf import PdfPages
from astropy.stats import sigma_clipped_stats
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.visualization import ZScaleInterval, AsinhStretch, ImageNormalize

import lsst.daf.butler as dafButler

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting
from common.camera_utils import pixel_to_focal

setup_plotting()

os.makedirs(output_dir, exist_ok=True)


<a id='functions'></a>
## Helper Functions

In [ ]:
def list_high_b_exposures(butler, collection, dataset_type, instrument, day_obs,
                          b_min):
    """List night exposures with galactic latitude |b| > b_min.

    RA/Dec are taken from the butler exposure record (tracking_ra/tracking_dec)
    and converted to galactic coordinates with astropy.

    Returns
    -------
    list[dict]
        Sorted by exposure id; each dict has id, ra, dec, gal_b, physical_filter,
        obs_reason, exptime.
    """
    refs = list(butler.query_datasets(
        dataset_type, collections=collection,
        where=f"instrument=\'{instrument}\' and exposure.day_obs={day_obs}",
        limit=None,
    ))
    exposures = sorted({r.dataId["exposure"] for r in refs})

    rows = []
    for exp in exposures:
        (rec,) = butler.registry.queryDimensionRecords(
            "exposure", where=f"instrument=\'{instrument}\' and exposure={exp}")
        if rec.tracking_ra is None or rec.tracking_dec is None:
            continue
        gal = SkyCoord(ra=rec.tracking_ra * u.deg, dec=rec.tracking_dec * u.deg,
                       frame="icrs").galactic
        rows.append(dict(
            id=exp, ra=rec.tracking_ra, dec=rec.tracking_dec, gal_b=float(gal.b.deg),
            physical_filter=rec.physical_filter, obs_reason=rec.observation_reason,
            exptime=rec.exposure_time,
        ))
    return [r for r in rows if abs(r["gal_b"]) > b_min]


def load_wfs_exposure(butler, collection, dataset_type, instrument, exposure, detector):
    """Load one post-ISR corner-WFS exposure object (carries image + detector)."""
    return butler.get(dataset_type, collections=collection,
                      dataId={"instrument": instrument, "exposure": exposure,
                              "detector": detector})


def build_radius_map(detector, image_shape):
    """Focal-plane radius (mm) for every pixel of a (possibly binned) image.

    The transform is identical for all visits, so this is computed once per
    detector and cached by the caller. Handles binning by comparing the image
    shape to the detector bounding box.
    """
    bbox = detector.getBBox()
    ny, nx = image_shape
    binx = bbox.getWidth() / nx
    biny = bbox.getHeight() / ny
    xs = bbox.getMinX() + (np.arange(nx) + 0.5) * binx - 0.5
    ys = bbox.getMinY() + (np.arange(ny) + 0.5) * biny - 0.5
    xx, yy = np.meshgrid(xs, ys)
    fpx, fpy = pixel_to_focal(xx.ravel(), yy.ravel(), detector)
    r = np.hypot(fpx, fpy).reshape(image_shape)
    return r.astype(np.float32)


def radial_profile(data, rmap, bin_edges, sigma=3.0, maxiters=5, min_pix=50):
    """Sigma-clipped mean flux in focal-plane radius bins.

    Returns (centers, means, counts) for bins with at least `min_pix` pixels.
    """
    rflat = rmap.ravel()
    fflat = data.ravel()
    idx = np.digitize(rflat, bin_edges) - 1
    centers, means, counts = [], [], []
    for k in range(len(bin_edges) - 1):
        sel = idx == k
        n = int(sel.sum())
        if n < min_pix:
            continue
        m, _, _ = sigma_clipped_stats(fflat[sel], sigma=sigma, maxiters=maxiters)
        centers.append(0.5 * (bin_edges[k] + bin_edges[k + 1]))
        means.append(float(m))
        counts.append(n)
    return np.array(centers), np.array(means), np.array(counts)


def normalize_profile(centers, means, r_lo, r_hi):
    """Divide a radial profile by its mean over [r_lo, r_hi] (set that window->1.0).

    Returns the normalised means (NaN-filled if no bins fall in the window).
    """
    if len(centers) == 0:
        return means
    in_win = (centers >= r_lo) & (centers <= r_hi)
    if not np.any(in_win):
        return np.full_like(means, np.nan)
    return means / np.mean(means[in_win])


def radial_sky_model(rmap, centers, means):
    """Per-pixel sky model: the radial profile interpolated to every pixel radius.

    Linear interpolation in radius; clamped to the profile endpoints outside the
    measured range. Returns an array the same shape as `rmap`.
    """
    if len(centers) == 0:
        return np.full_like(rmap, np.nan, dtype=float)
    return np.interp(rmap, centers, means).astype(float)


def donut_mask(data, *, method="radial", sigma=3.0, maxiters=5,
               rmap=None, profile=None):
    """Mask of donut/star (high) pixels.

    method="global": mask pixels above mean + sigma*std (sigma-clipped, whole chip).
    method="radial": subtract the radial sky model, then mask pixels whose residual
                     exceeds mean + sigma*std of the (sigma-clipped) residual. This
                     keeps the radial sky gradient out of the mask.

    Returns (mask, info) where info has the sky level / residual std used.
    """
    if method == "global":
        mean, _, std = sigma_clipped_stats(data, sigma=sigma, maxiters=maxiters)
        return data > mean + sigma * std, {"mean": float(mean), "std": float(std)}
    if method == "radial":
        if rmap is None or profile is None:
            raise ValueError("radial masking needs rmap and profile")
        centers, means = profile[0], profile[1]
        model = radial_sky_model(rmap, centers, means)
        resid = data - model
        rmean, _, rstd = sigma_clipped_stats(resid, sigma=sigma, maxiters=maxiters)
        return resid > rmean + sigma * rstd, {"resid_mean": float(rmean),
                                              "resid_std": float(rstd)}
    raise ValueError(f"unknown mask_method {method!r}")


def make_norm(data, mode="sky", *, sky_lo_sigma=2.0, sky_hi_sigma=8.0,
              zscale_contrast=0.25, zscale_nsamples=100_000,
              pct_low=1.0, pct_high=99.0, asinh_a=0.1):
    """ImageNormalize (asinh stretch) for a sky-dominated frame."""
    finite = data[np.isfinite(data)]
    if finite.size == 0:
        vmin, vmax = 0.0, 1.0
    elif mode == "sky":
        _, med, std = sigma_clipped_stats(finite, sigma=3.0, maxiters=5)
        vmin, vmax = med - sky_lo_sigma * std, med + sky_hi_sigma * std
    elif mode == "zscale":
        vmin, vmax = ZScaleInterval(contrast=zscale_contrast,
                                    n_samples=zscale_nsamples).get_limits(finite)
    elif mode == "percentile":
        vmin, vmax = np.nanpercentile(finite, [pct_low, pct_high])
    else:
        raise ValueError(f"unknown stretch mode {mode!r}")
    if not np.isfinite(vmax) or vmax <= vmin:
        vmax = vmin + 1.0
    return ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch(a=asinh_a))


def _grid_fig(rafts, img_shape, panel_width):
    """Create a tight 4x2 figure sized to the image aspect ratio."""
    h, w = img_shape
    panel_height = panel_width * h / w
    fig, axes = plt.subplots(len(rafts), 2,
                             figsize=(2 * panel_width, len(rafts) * panel_height),
                             constrained_layout=True)
    axes = np.atleast_2d(axes)
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, wspace=0.01, hspace=0.04)
    return fig, axes


<a id='data'></a>
## Data Access

In [ ]:
butler = dafButler.Butler(butler_repo, collections=collection)
print("Repo:", butler_repo)
print("Collection:", collection)
print("Dataset type:", dataset_type)


<a id='select'></a>
## Select High-Galactic-Latitude Visits

In [ ]:
high_b = list_high_b_exposures(butler, collection, dataset_type, instrument,
                               day_obs, b_min)
print(f"day_obs {day_obs}: {len(high_b)} visits with |b| > {b_min} deg\n")
print(f"{'exposure':>14} {'gal_b':>7} {'filter':>9} {'exptime':>8}  obs_reason")
for r in high_b:
    print(f"{r['id']:>14} {r['gal_b']:>7.1f} {str(r['physical_filter']):>9} "
          f"{r['exptime']:>8.1f}  {r['obs_reason']}")

visits = [r["id"] for r in high_b][start_index:]
if max_visits is not None:
    visits = visits[:max_visits]
print(f"\nRendering {len(visits)} visit(s):", visits)


<a id='radius'></a>
## Focal-Plane Radius Maps

In [ ]:
# The focal-plane geometry is identical for every visit, so build the per-pixel
# radius map once per detector (cached) from the first visit's exposures.
radius_maps = {}     # detector id -> radius map (mm)
detectors = {}       # detector id -> afw Detector (for bookkeeping)

if visits:
    v0 = visits[0]
    for det_id in WFS_DETECTORS:
        exp = load_wfs_exposure(butler, collection, dataset_type, instrument,
                                v0, det_id)
        det = exp.getDetector()
        detectors[det_id] = det
        radius_maps[det_id] = build_radius_map(det, exp.image.array.shape)
    rmin = min(float(rm.min()) for rm in radius_maps.values())
    rmax = max(float(rm.max()) for rm in radius_maps.values())
    radius_bin_edges = np.arange(rmin, rmax + radial_bin_mm, radial_bin_mm)
    fine_bin_edges = np.arange(rmin, r_max_donut + fine_bin_mm, fine_bin_mm)
    print(f"Radius range across CWFS: {rmin:.1f}–{rmax:.1f} mm")
    print(f"  page 3: {len(radius_bin_edges)-1} bins of {radial_bin_mm} mm "
          f"({rmin:.1f}–{rmax:.1f} mm)")
    print(f"  page 4: {len(fine_bin_edges)-1} bins of {fine_bin_mm} mm "
          f"({rmin:.1f}–{r_max_donut} mm), normalised over "
          f"{norm_r_lo}–{norm_r_hi} mm")


<a id='analysis'></a>
## Analysis & PDF (4 pages per visit)

In [ ]:
norm_kwargs = dict(
    sky_lo_sigma=sky_lo_sigma, sky_hi_sigma=sky_hi_sigma,
    zscale_contrast=zscale_contrast, zscale_nsamples=zscale_nsamples,
    pct_low=pct_low, pct_high=pct_high, asinh_a=asinh_a,
)
rafts = sorted({n.split("_")[0] for n in WFS_DETECTORS.values()})
name_to_det = {v: k for k, v in WFS_DETECTORS.items()}
red_cmap = ListedColormap([mask_color])
det_colors = {det: c for det, c in
              zip(WFS_DETECTORS, plt.cm.viridis(np.linspace(0, 1, len(WFS_DETECTORS))))}


def page_images(visit, images, gal_b):
    """Page 1: sky-stretched images."""
    fig, axes = _grid_fig(rafts, next(iter(images.values())).shape, panel_width)
    for i, raft in enumerate(rafts):
        for j, sw in enumerate(("SW0", "SW1")):
            ax = axes[i, j]; name = f"{raft}_{sw}"; data = images[name_to_det[name]]
            ax.imshow(data, origin="lower", cmap=cmap,
                      norm=make_norm(data, mode=stretch_mode, **norm_kwargs),
                      aspect="equal", interpolation="nearest")
            ax.set_title(f"{name} ({name_to_det[name]})  med={np.nanmedian(data):.0f}",
                         fontsize=title_fontsize)
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"Visit {visit}  |b|={abs(gal_b):.1f}°  — sky stretch",
                 fontsize=title_fontsize + 3)
    return fig


def page_masked(visit, images, masks, gal_b):
    """Page 2: donut/star pixels highlighted (sky-isolation diagnostic)."""
    fig, axes = _grid_fig(rafts, next(iter(images.values())).shape, panel_width)
    for i, raft in enumerate(rafts):
        for j, sw in enumerate(("SW0", "SW1")):
            ax = axes[i, j]; name = f"{raft}_{sw}"; det = name_to_det[name]
            data, mask = images[det], masks[det]
            ax.imshow(data, origin="lower", cmap=cmap,
                      norm=make_norm(data, mode=stretch_mode, **norm_kwargs),
                      aspect="equal", interpolation="nearest")
            overlay = np.ma.masked_where(~mask, np.ones_like(data))
            ax.imshow(overlay, origin="lower", cmap=red_cmap, vmin=0, vmax=1,
                      alpha=0.6, aspect="equal", interpolation="nearest")
            ax.set_title(f"{name} ({det})  masked={100*mask.mean():.1f}%",
                         fontsize=title_fontsize)
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"Visit {visit}  |b|={abs(gal_b):.1f}°  — donut/star pixels masked "
                 f"({mask_color}, method={mask_method}, {clip_sigma}σ)",
                 fontsize=title_fontsize + 3)
    return fig


def page_profiles(visit, profiles, gal_b):
    """Page 3: clipped-mean sky flux vs focal-plane radius, 8 curves (coarse)."""
    fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
    for det, name in WFS_DETECTORS.items():
        r, m, _ = profiles[det]
        if len(r):
            ax.plot(r, m, "-o", ms=3, color=det_colors[det], label=f"{name} ({det})")
    ax.set_xlabel("Focal-plane radius  [mm]")
    ax.set_ylabel(f"{clip_sigma}σ-clipped mean flux  [e-]")
    ax.set_title(f"Visit {visit}  |b|={abs(gal_b):.1f}°  — sky level vs radius")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    return fig


def page_profiles_norm(visit, profiles_fine, gal_b):
    """Page 4: fine-binned profile out to r_max_donut, normalised per CCD."""
    fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
    for det, name in WFS_DETECTORS.items():
        r, m, _ = profiles_fine[det]
        if len(r):
            mn = normalize_profile(r, m, norm_r_lo, norm_r_hi)
            ax.plot(r, mn, "-", lw=1.2, color=det_colors[det], label=f"{name} ({det})")
    ax.axhline(1.0, color="0.5", lw=0.8, ls=":")
    ax.axvspan(norm_r_lo, norm_r_hi, color="0.85", alpha=0.5, zorder=0,
               label=f"norm window {norm_r_lo:.0f}–{norm_r_hi:.0f} mm")
    ax.set_xlim(None, r_max_donut)
    ax.set_xlabel("Focal-plane radius  [mm]")
    ax.set_ylabel(f"Normalised {clip_sigma}σ-clipped mean  (=1 over "
                  f"{norm_r_lo:.0f}–{norm_r_hi:.0f} mm)")
    ax.set_title(f"Visit {visit}  |b|={abs(gal_b):.1f}°  — normalised sky vs radius "
                 f"(≤{r_max_donut} mm = 1.75°, {fine_bin_mm} mm bins)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    return fig


pdf_path = Path(output_pdf) if output_pdf else \
    Path(output_dir) / f"wfs_sky_radius_{day_obs}.pdf"

with PdfPages(pdf_path) as pdf:
    for visit in visits:
        gal_b = next(r["gal_b"] for r in high_b if r["id"] == visit)
        images, masks, profiles, profiles_fine = {}, {}, {}, {}
        for det_id in WFS_DETECTORS:
            exp = load_wfs_exposure(butler, collection, dataset_type, instrument,
                                    visit, det_id)
            data = np.asarray(exp.image.array, dtype=float)
            images[det_id] = data
            # Coarse profile (page 3); also serves as the sky model for radial masking.
            profiles[det_id] = radial_profile(data, radius_maps[det_id],
                                              radius_bin_edges, clip_sigma,
                                              clip_maxiters)
            # Fine profile out to the donut-acceptance radius (page 4).
            profiles_fine[det_id] = radial_profile(data, radius_maps[det_id],
                                                   fine_bin_edges, clip_sigma,
                                                   clip_maxiters)
            masks[det_id], _ = donut_mask(
                data, method=mask_method, sigma=clip_sigma, maxiters=clip_maxiters,
                rmap=radius_maps[det_id], profile=profiles[det_id])

        for builder, args in ((page_images, (images, gal_b)),
                              (page_masked, (images, masks, gal_b)),
                              (page_profiles, (profiles, gal_b)),
                              (page_profiles_norm, (profiles_fine, gal_b))):
            fig = builder(visit, *args)
            pdf.savefig(fig, dpi=dpi)
            plt.show()
        print(f"visit {visit}: 4 pages written")

print(f"\nWrote {4*len(visits)} page(s) to {pdf_path}")
